# 03 — Especificación del modelado y evidencia

Este cuaderno no entrena modelos ni crea un Parquet artificialmente completo. Su propósito es elegir un objetivo defendible, fijar el momento de predicción, auditar fuga de información y documentar el sustento de cada decisión antes de preparar datos de modelado.

> Los registros CEM describen casos atendidos; no miden prevalencia poblacional. Cualquier modelo futuro será exploratorio y no debe automatizar decisiones de protección, legales o de acceso a servicios.

## Evidencia metodológica

La bitácora completa está en `registro_evidencia_metodologica.md`. Sus decisiones centrales son: ausencia con semántica explícita, separación temporal para evaluar generalización, transformaciones aprendidas solo en entrenamiento y métricas por clase/calibración. Referencias: [He (2010)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2818781/), [Wijesuriya et al. (2020)](https://pmc.ncbi.nlm.nih.gov/articles/PMC7586995/), [Ramspek et al. (2021)](https://pmc.ncbi.nlm.nih.gov/articles/PMC7857818/), [Kaufman et al. (2012)](https://dl.acm.org/doi/10.1145/2382577.2382579) y [Kuhn et al. (2025)](https://pmc.ncbi.nlm.nih.gov/articles/PMC13293568/).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'salidas_limpieza' / 'base_historica_2020_2025_ubigeo.parquet'
OUTPUT_DIR = BASE_DIR / 'salidas_modelado'
OUTPUT_DIR.mkdir(exist_ok=True)
DATE_FORMAT = '%m/%d/%Y'
# Diccionario de targets validado por el responsable del proyecto el 2026-08-27.
TARGET_LABELS = {
    'TIPO_VIOLENCIA': {'0': 'Económica', '1': 'Psicológica', '2': 'Física', '3': 'Sexual'},
    'NIVEL_DE_RIESGO_VICTIMA': {'1': 'Leve', '2': 'Moderado', '3': 'Severo'},
    'CONDICION': {'1': 'Nuevo', '2': 'Reingreso', '3': 'Reincidente', '4': 'Derivado', '5': 'Continuador'},
}
assert DATA_PATH.exists(), 'Primero ejecute el notebook 02 para generar la base histórica.'

df = pd.read_parquet(DATA_PATH)
df['anio_ingreso'] = pd.to_datetime(df['FECHA_INGRESO'], format=DATE_FORMAT, errors='coerce').dt.year
print(f'Registros: {len(df):,}; variables: {df.shape[1]}')
print('Años:', sorted(df['anio_ingreso'].dropna().unique()))

Registros: 936,835; variables: 34
Años: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


## 1. Factibilidad estadística de los targets

Los códigos se muestran tal como llegaron en la fuente. No se asignan etiquetas textuales hasta verificar el diccionario oficial de variables. Una categoría rara no se elimina automáticamente: su tratamiento depende de la pregunta, el costo de error y la definición sustantiva.

In [2]:
def resumen_target(data, target):
    tabla = data[target].value_counts(dropna=False).rename_axis('codigo').reset_index(name='n')
    tabla['pct'] = (tabla['n'] / len(data) * 100).round(2)
    tabla['codigo'] = tabla['codigo'].astype('string')
    tabla['etiqueta'] = tabla['codigo'].map(TARGET_LABELS[target]).fillna('Código no documentado')
    tabla['target'] = target
    return tabla[['target', 'codigo', 'etiqueta', 'n', 'pct']]

targets_principales = ['NIVEL_DE_RIESGO_VICTIMA', 'TIPO_VIOLENCIA']
target_summary = pd.concat([resumen_target(df, target) for target in targets_principales], ignore_index=True)
target_summary.to_csv(OUTPUT_DIR / 'distribucion_targets.csv', index=False, encoding='utf-8-sig')
display(target_summary)

for target in targets_principales:
    print(f'\n{target}: distribución anual (%)')
    display(pd.crosstab(df['anio_ingreso'], df[target], normalize='index').mul(100).round(2))

,target,codigo,etiqueta,n,pct
0,NIVEL_DE_RIESGO_VICTIMA,2,Moderado,485470,51.82
1,NIVEL_DE_RIESGO_VICTIMA,3,Severo,249861,26.67
2,NIVEL_DE_RIESGO_VICTIMA,1,Leve,201504,21.51
3,TIPO_VIOLENCIA,1,Psicológica,415308,44.33
4,TIPO_VIOLENCIA,2,Física,358486,38.27
5,TIPO_VIOLENCIA,3,Sexual,159066,16.98
6,TIPO_VIOLENCIA,0,Económica,3975,0.42



NIVEL_DE_RIESGO_VICTIMA: distribución anual (%)


NIVEL_DE_RIESGO_VICTIMA,1,2,3
anio_ingreso,,,
2020,25.56,50.79,23.64
2021,24.93,51.93,23.14
2022,22.53,53.15,24.31
2023,20.34,52.07,27.59
2024,18.76,51.55,29.69
2025,18.42,51.21,30.37



TIPO_VIOLENCIA: distribución anual (%)


TIPO_VIOLENCIA,0,1,2,3
anio_ingreso,,,,
2020,0.46,48.91,38.54,12.09
2021,0.39,46.33,39.56,13.71
2022,0.45,43.21,38.60,17.74
2023,0.41,42.74,38.32,18.54
2024,0.41,42.56,37.80,19.22
2025,0.43,43.65,36.93,18.98


In [3]:
candidate_targets = pd.DataFrame([
    ['NIVEL_DE_RIESGO_VICTIMA', 'Leve / Moderado / Severo; 0% nulos', 'Replicar retrospectivamente la clasificación registrada', 'Probable evaluación concurrente; prohibido afirmar predicción prospectiva sin validar el protocolo', 'Principal para factibilidad, pendiente de auditoría de fuga'],
    ['CONDICION', 'Nuevo / Reingreso / Reincidente / Derivado / Continuador; 0% nulos', 'Caracterizar trayectoria administrativa del caso', 'Reincidente incluye necesidad de patrocinio legal; puede contener información posterior o de servicio', 'Candidata para análisis; no aprobada como predictor prospectivo'],
    ['TIPO_VIOLENCIA', 'Económica / Psicológica / Física / Sexual; 0% nulos; Económica 0.42%', 'Describir o clasificar el tipo registrado', 'Puede estar definido por indicadores de violencia registrados en la misma ficha', 'Secundario; diccionario confirmado; evaluar clase económica rara sin excluirla por defecto'],
    ['INTERPUSO_DENUNCIA', '2 clases; 0% nulos', 'Estudiar un resultado procedimental', 'Puede ocurrir después de la evaluación inicial', 'Solo con instante de predicción previo confirmado'],
    ['CASO_TENTATIVA_DE_FEMINICIDIO', '4.59% nulos; categoría dominante 99.82%', 'Evento raro', 'Desbalance extremo y ausencia no resuelta', 'No recomendado en primera fase'],
], columns=['target', 'calidad_observada', 'uso_posible', 'riesgo_metodologico', 'estado'])
candidate_targets.to_csv(OUTPUT_DIR / 'evaluacion_targets.csv', index=False, encoding='utf-8-sig')
display(candidate_targets)

,target,calidad_observada,uso_posible,riesgo_metodologico,estado
0,NIVEL_DE_RIESGO_VICTIMA,Leve / Moderado / Severo; 0% nulos,Replicar retrospectivamente la clasificación r...,Probable evaluación concurrente; prohibido afi...,"Principal para factibilidad, pendiente de audi..."
1,CONDICION,Nuevo / Reingreso / Reincidente / Derivado / C...,Caracterizar trayectoria administrativa del caso,Reincidente incluye necesidad de patrocinio le...,Candidata para análisis; no aprobada como pred...
2,TIPO_VIOLENCIA,Económica / Psicológica / Física / Sexual; 0% ...,Describir o clasificar el tipo registrado,Puede estar definido por indicadores de violen...,Secundario; diccionario confirmado; evaluar cl...
3,INTERPUSO_DENUNCIA,2 clases; 0% nulos,Estudiar un resultado procedimental,Puede ocurrir después de la evaluación inicial,Solo con instante de predicción previo confirmado
4,CASO_TENTATIVA_DE_FEMINICIDIO,4.59% nulos; categoría dominante 99.82%,Evento raro,Desbalance extremo y ausencia no resuelta,No recomendado en primera fase


## 2. Puerta contra fuga de información

La clasificación de riesgo CEM se realiza dentro de la atención, según el contexto institucional de los CEM. Por tanto, antes de utilizar `NIVEL_DE_RIESGO_VICTIMA` como target se debe decidir si el objetivo es **replicar retrospectivamente una etiqueta registrada** o **predecir antes de la valoración**. Solo el segundo caso exige que cada predictor esté disponible antes de dicha valoración.

Esta distinción es esencial: alta precisión para reproducir una evaluación ya realizada no demuestra utilidad para anticiparla.

In [4]:
# Completar únicamente después de validar el protocolo/ficha de registro con la fuente responsable.
TARGET_SELECCIONADO = None  # Ejemplo: 'NIVEL_DE_RIESGO_VICTIMA'
MOMENTO_PREDICCION = None  # Ejemplo: 'antes_de_valoracion_de_riesgo'
OBJETIVO_USO = None  # Ejemplo: 'analisis_retrospectivo' o 'apoyo_exploratorio'

# Solo columnas comprobadas como disponibles en el momento anterior. No usar la lista por defecto como autorización.
FEATURES_APROBADAS = []

# Variables que nunca pueden entrar si el target es nivel de riesgo: target mismo y resultados/acciones posteriores.
BLOQUEADAS_RIESGO = {
    'NIVEL_DE_RIESGO_VICTIMA', 'INTERPUSO_DENUNCIA', 'DESEA_PATROCINIO_LEGAL',
    'CONTINUA_RECIBIENDO_TRATAMIENTO', 'CUENTA_MEDIDAS_PROTECCION',
    'CUENTA_MEDIDAS_CAUTELARES', 'DESEA_DENUNCIAR',
}

def validar_especificacion(target, momento, objetivo, features):
    problemas = []
    if target not in df.columns:
        problemas.append('Seleccione un target existente.')
    if not momento:
        problemas.append('Defina el momento de predicción.')
    if not objetivo:
        problemas.append('Defina el uso analítico permitido.')
    if not features:
        problemas.append('Apruebe variables con evidencia de disponibilidad temporal.')
    if target == 'NIVEL_DE_RIESGO_VICTIMA' and set(features).intersection(BLOQUEADAS_RIESGO):
        problemas.append('Hay variables bloqueadas por posible fuga para el target de riesgo.')
    if not set(features).issubset(df.columns):
        problemas.append('Hay variables aprobadas que no existen en la base.')
    return problemas

problemas = validar_especificacion(TARGET_SELECCIONADO, MOMENTO_PREDICCION, OBJETIVO_USO, FEATURES_APROBADAS)
print('Especificación lista.' if not problemas else 'Pendiente: ' + ' | '.join(problemas))

Pendiente: Seleccione un target existente. | Defina el momento de predicción. | Defina el uso analítico permitido. | Apruebe variables con evidencia de disponibilidad temporal.


## 3. Criterio para el Parquet final sin nulos

El Parquet final se genera **después** de superar la puerta anterior. Las transformaciones deben ajustarse únicamente sobre el entrenamiento 2020–2023, aplicarse sin reentrenar a 2024 y reservar 2025 como prueba final. Para cada predictor debe registrarse una de estas opciones: categoría explícita “no registrado” más indicador de ausencia, imputación aprendida solo con entrenamiento, o exclusión justificada.

No se deben calcular medianas, codificaciones, selección de variables, SMOTE ni escalado usando 2024–2025. Si se usa remuestreo, se aplica solo dentro de cada pliegue de entrenamiento y se compara contra el baseline sin remuestreo.

In [5]:
plan_final = pd.DataFrame([
    ['Entrenamiento', '2020–2023', 'Ajustar imputación/codificación/selección únicamente aquí'],
    ['Validación temporal', '2024', 'Elegir hiperparámetros y comparar baseline vs remuestreo'],
    ['Prueba final', '2025', 'Una sola evaluación final; no retocar decisiones después'],
    ['Métricas', 'Por clase + calibración', 'Macro-F1, precisión, recall, matriz de confusión, Brier/calibración cuando corresponda'],
    ['Equidad', 'Subgrupos predefinidos', 'Reportar desempeño por territorio/edad/sexo solo con tamaños seguros y sin reidentificación'],
], columns=['etapa', 'periodo', 'regla'])
plan_final.to_csv(OUTPUT_DIR / 'plan_validacion_temporal.csv', index=False, encoding='utf-8-sig')
display(plan_final)

,etapa,periodo,regla
0,Entrenamiento,2020–2023,Ajustar imputación/codificación/selección únic...
1,Validación temporal,2024,Elegir hiperparámetros y comparar baseline vs ...
2,Prueba final,2025,Una sola evaluación final; no retocar decision...
3,Métricas,Por clase + calibración,"Macro-F1, precisión, recall, matriz de confusi..."
4,Equidad,Subgrupos predefinidos,Reportar desempeño por territorio/edad/sexo so...
